In [1]:
!pip install gradio

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import tensorflow as tf
import cv2
import gradio as gr

# 1. Path to your uploaded TFLite model in Google Drive
model_path = '/content/drive/MyDrive/Aidataset/ecoscan_mobilenet (1).tflite'

# 2. Load the TFLite model and allocate memory
interpreter = tf.lite.Interpreter(model_path=model_path)
interpreter.allocate_tensors()

# Get the specific input/output requirements of your MobileNet model
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
input_shape = input_details[0]['shape']

# 3. Define the exact categories from your dataset
labels = [
    'battery', 'glass', 'metal', 'organic_waste',
    'paper_cardboard', 'plastic', 'textiles', 'trash'
]

# 4. The Prediction Logic
def classify_waste(image):
    if image is None:
        return "Please upload an image."

    # Get the height and width expected by your TFLite model
    expected_height = input_shape[1]
    expected_width = input_shape[2]

    # Resize the image to match MobileNet's expected input
    img_resized = cv2.resize(image, (expected_width, expected_height))

    # Convert to float32 and normalize the pixel values to [0, 1]
    input_data = np.array(img_resized, dtype=np.float32) / 255.0

    # Add a batch dimension so the shape becomes [1, width, height, channels]
    input_data = np.expand_dims(input_data, axis=0)

    # Pass the image into the TFLite model
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()

    # Extract the prediction probabilities
    output_data = interpreter.get_tensor(output_details[0]['index'])[0]

    # Find the category with the highest probability
    top_index = np.argmax(output_data)
    confidence = output_data[top_index] * 100

    return f"Predicted Category: {labels[top_index].upper()} \nConfidence: {confidence:.2f}%"

# 5. Build and Launch the Gradio Interface
interface = gr.Interface(
    fn=classify_waste,
    inputs=gr.Image(type="numpy", label="Upload a Photo of Waste"),
    outputs=gr.Textbox(label="EcoScan Result"),
    title="🌍 EcoScan: MobileNet Waste Classifier",
    description="Upload an image to see the MobileNet TFLite model's classification.",
    allow_flagging="never"
)

# Launch the app and generate the public web link
interface.launch(share=True)

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e610c13d4a50782b62.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
